# Optimized Hybrid Retrieval — DevRev Search Benchmark

Hybrid retrieval pipeline: Dense + BM25 → Weighted RRF → Sibling Expansion → Cross-Encoder Rerank.

Optimized from ablation findings: BM25 dominates dense (R@10: 14.63 vs 4.85), so BM25 gets 2x weight in RRF. Reranker is the biggest lever (+8% R@10). Wider candidate pools and article-sibling expansion feed more signal to the reranker.

**Systems:**

| Component | Type | Open Source | Details |
|-----------|------|-------------|---------|
| [Snowflake Arctic Embed L v2.0](https://huggingface.co/Snowflake/snowflake-arctic-embed-l-v2.0) | Dense embedding model (1024d) | Yes, Apache 2.0 | Encodes queries and documents for semantic similarity search |
| [BM25 Okapi (rank-bm25)](https://github.com/dorianbrown/rank_bm25) | Sparse term-frequency retrieval | Yes, Apache 2.0 | Keyword-based scoring over tokenized corpus |
| [FAISS FlatIP](https://github.com/facebookresearch/faiss) | Vector similarity search | Yes, MIT | Exact inner-product nearest neighbor index by Meta |
| [BAAI BGE-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3) | Cross-encoder reranker | Yes, Apache 2.0 | Re-scores query-document pairs for precision |
| [DevRev Search](https://huggingface.co/datasets/devrev/search) | IR evaluation benchmark | Yes | ~65K doc chunks, 291 annotated + 92 test queries |

In [ ]:
import os, json, pickle, re, time, warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import faiss
from rank_bm25 import BM25Okapi
from tqdm import tqdm
from datasets import load_dataset

warnings.filterwarnings('ignore')

# --- Config ---
EMBED_MODEL_NAME = "Snowflake/snowflake-arctic-embed-l-v2.0"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
INDEX_DIR = "faiss_index_local"

# Optimized parameters
DENSE_K = 200
BM25_K = 200
RRF_K = 60
BM25_WEIGHT = 2.0      # BM25 contributes 2x in RRF (since it outperforms dense)
DENSE_WEIGHT = 1.0
RERANK_N = 150          # more candidates for reranker
DOC_TRUNC = 768         # longer context for reranker
SIBLING_HOPS = 1        # include neighboring chunks from same article

## 1. Load Everything

In [ ]:
# FAISS index + doc mapping
index = faiss.read_index(os.path.join(INDEX_DIR, "knowledge_base_flat.index"))
with open(os.path.join(INDEX_DIR, "doc_mapping.pkl"), "rb") as f:
    mapping = pickle.load(f)

doc_ids = mapping["doc_ids"]
documents = mapping["documents"]
doc_titles = mapping["doc_titles"]
doc_texts = mapping["doc_texts"]

# Build article-sibling lookup: doc_id -> list of sibling doc indices
article_to_indices = defaultdict(list)
for i, did in enumerate(doc_ids):
    art_id = did.split("_KNOWLEDGE_NODE")[0] if "_KNOWLEDGE_NODE" in did else did
    article_to_indices[art_id].append(i)

idx_to_article_indices = {}
for art_id, indices in article_to_indices.items():
    for idx in indices:
        idx_to_article_indices[idx] = indices

print(f"FAISS index: {index.ntotal:,} vectors (dim={index.d})")
print(f"Documents: {len(documents):,}")
print(f"Articles: {len(article_to_indices):,}")

In [ ]:
# Datasets
test_queries = load_dataset("devrev/search", "test_queries", split="test")
annotated_queries = load_dataset("devrev/search", "annotated_queries", split="train")
print(f"Test queries: {len(test_queries)}, Annotated queries: {len(annotated_queries)}")

In [ ]:
# Models
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device, trust_remote_code=True)
print(f"Embed model loaded: dim={embed_model.get_sentence_embedding_dimension()}")

reranker = CrossEncoder(RERANKER_MODEL_NAME, device=device, trust_remote_code=True)
print(f"Reranker loaded: {RERANKER_MODEL_NAME}")

In [ ]:
# BM25
with open("bm25_local.pkl", "rb") as f:
    bm25 = pickle.load(f)
print(f"BM25 index loaded: {bm25.corpus_size:,} docs")

## 2. Core Functions

In [ ]:
def tokenize(text: str) -> list[str]:
    return re.findall(r'\w+', text.lower())


def embed_queries(queries: list[str]) -> np.ndarray:
    texts = [QUERY_PREFIX + q for q in queries]
    embs = embed_model.encode(texts, batch_size=64, show_progress_bar=False,
                               convert_to_numpy=True, normalize_embeddings=True)
    return embs.astype(np.float32)


def embed_single(query: str) -> np.ndarray:
    return embed_queries([query])[0]


def search_dense(query_emb: np.ndarray, k: int = 200) -> list[tuple[int, float]]:
    q = query_emb.reshape(1, -1)
    scores, indices = index.search(q, k)
    return [(int(i), float(s)) for s, i in zip(scores[0], indices[0]) if i >= 0]


def search_bm25(query: str, k: int = 200) -> list[tuple[int, float]]:
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)
    top_k = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top_k if scores[i] > 0]

## 3. Weighted RRF + Article Sibling Expansion

In [ ]:
def weighted_rrf(ranked_lists: list, weights: list[float], k: int = 60) -> list[tuple[int, float]]:
    """RRF with per-source weights. Higher weight = more influence."""
    rrf_scores = defaultdict(float)
    for rlist, w in zip(ranked_lists, weights):
        for rank, (doc_idx, _) in enumerate(rlist):
            rrf_scores[doc_idx] += w / (k + rank + 1)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)


def expand_siblings(candidates: list[tuple[int, float]], max_hops: int = 1) -> list[tuple[int, float]]:
    """For each candidate, add neighboring chunks from the same article.
    Siblings get a discounted score."""
    seen = {idx for idx, _ in candidates}
    expanded = list(candidates)

    for idx, score in candidates:
        siblings = idx_to_article_indices.get(idx, [])
        for sib_idx in siblings:
            if sib_idx not in seen:
                # Only add if within max_hops chunk distance
                if abs(sib_idx - idx) <= max_hops:
                    expanded.append((sib_idx, score * 0.5))
                    seen.add(sib_idx)

    return expanded


# Quick test
q_emb = embed_single("How do I set up AirSync?")
dense = search_dense(q_emb, k=5)
sparse = search_bm25("How do I set up AirSync?", k=5)
fused = weighted_rrf([dense, sparse], [DENSE_WEIGHT, BM25_WEIGHT], k=RRF_K)
expanded = expand_siblings(fused[:10])
print(f"Fused: {len(fused)} docs, After sibling expansion: {len(expanded)} docs")
for idx, score in fused[:5]:
    print(f"  {score:.4f} | {doc_ids[idx]} | {doc_titles[idx][:60]}")

## 4. Reranker

In [ ]:
def rerank_candidates(query: str, candidates: list[tuple[int, float]],
                      top_k: int = 50, max_cands: int = RERANK_N,
                      doc_trunc: int = DOC_TRUNC) -> list[tuple[int, float]]:
    if not candidates:
        return []
    candidates = candidates[:max_cands]
    pairs = [(query, documents[idx][:doc_trunc]) for idx, _ in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False)
    scored = [(candidates[i][0], float(scores[i])) for i in range(len(candidates))]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

## 5. Optimized Pipeline

In [ ]:
def optimized_retrieve(query: str, top_k: int = 50) -> list[tuple[int, float]]:
    """Dense + BM25 -> Weighted RRF -> Sibling expansion -> Rerank."""
    q_emb = embed_single(query)
    dense_results = search_dense(q_emb, k=DENSE_K)
    bm25_results = search_bm25(query, k=BM25_K)

    # Weighted RRF (BM25 gets 2x weight)
    fused = weighted_rrf([dense_results, bm25_results],
                         [DENSE_WEIGHT, BM25_WEIGHT], k=RRF_K)

    # Expand with article siblings
    expanded = expand_siblings(fused[:RERANK_N], max_hops=SIBLING_HOPS)

    # Rerank
    return rerank_candidates(query, expanded, top_k=top_k)


# Test
results = optimized_retrieve("end customer organization name not appearing in ticket", top_k=5)
print("Optimized results:")
for rank, (idx, score) in enumerate(results):
    print(f"  [{rank+1}] {score:.4f} | {doc_ids[idx]} | {doc_titles[idx][:60]}")

## 6. Evaluation

In [ ]:
def evaluate_pipeline(
    pipeline_fn,
    annotated_data,
    top_k_list=(10, 50),
    max_queries: int = None,
    label: str = "Pipeline",
) -> dict:
    items = list(annotated_data)
    if max_queries:
        items = items[:max_queries]

    max_k = max(top_k_list)
    results_by_k = {k: {"recalls": [], "precisions": [], "mrrs": []} for k in top_k_list}
    per_query = []

    for item in tqdm(items, desc=f"Eval: {label}"):
        query = item["query"]
        golden_ids = {r["id"] for r in item["retrievals"]}

        ranked = pipeline_fn(query, top_k=max_k)
        retrieved_ids = [doc_ids[idx] for idx, _ in ranked]

        for k in top_k_list:
            top_ids = retrieved_ids[:k]
            hits = sum(1 for rid in top_ids if rid in golden_ids)
            recall = hits / len(golden_ids) if golden_ids else 0
            precision = hits / k
            results_by_k[k]["recalls"].append(recall)
            results_by_k[k]["precisions"].append(precision)

            rr = 0.0
            for rank_pos, rid in enumerate(top_ids):
                if rid in golden_ids:
                    rr = 1.0 / (rank_pos + 1)
                    break
            results_by_k[k]["mrrs"].append(rr)

        per_query.append({
            "query_id": item["query_id"], "query": query,
            "golden_count": len(golden_ids),
            "hits@10": sum(1 for rid in retrieved_ids[:10] if rid in golden_ids),
            "hits@50": sum(1 for rid in retrieved_ids[:50] if rid in golden_ids),
        })

    print(f"\n{'='*70}")
    print(f"  {label} -- {len(items)} queries")
    print(f"{'='*70}")
    metrics = {"label": label, "n_queries": len(items)}
    for k in top_k_list:
        r = np.mean(results_by_k[k]["recalls"]) * 100
        p = np.mean(results_by_k[k]["precisions"]) * 100
        m = np.mean(results_by_k[k]["mrrs"])
        print(f"  Recall@{k}: {r:.2f}%   Precision@{k}: {p:.2f}%   MRR@{k}: {m:.4f}")
        metrics[f"recall@{k}"] = r
        metrics[f"precision@{k}"] = p
        metrics[f"mrr@{k}"] = m

    metrics["per_query"] = per_query
    return metrics

In [ ]:
# Run full evaluation on all 291 annotated queries
optimized_metrics = evaluate_pipeline(optimized_retrieve, annotated_queries,
                                       label="Optimized (weighted RRF + siblings + rerank)")

In [ ]:
# Compare with baseline results
print("\n" + "=" * 90)
print("COMPARISON: Baseline vs Optimized")
print("=" * 90)
print(f"{'Method':<50} {'R@10':>8} {'R@50':>8} {'MRR@10':>8}")
print("-" * 90)

baselines = [
    ("Baseline: Dense Only",              4.85,  9.16, 0.0959),
    ("Baseline: BM25 Only",              14.63, 28.38, 0.2756),
    ("Baseline: Dense+BM25 RRF",         12.56, 27.67, 0.2330),
    ("Baseline: Dense+BM25+Reranker",    20.66, 31.23, 0.4272),
]
for name, r10, r50, mrr in baselines:
    print(f"{name:<50} {r10:>7.2f}% {r50:>7.2f}% {mrr:>8.4f}")

om = optimized_metrics
print(f"{'>>> Optimized (this notebook)':<50} {om['recall@10']:>7.2f}% {om['recall@50']:>7.2f}% {om['mrr@10']:>8.4f}")
print("=" * 90)

delta_r10 = om['recall@10'] - 20.66
delta_r50 = om['recall@50'] - 31.23
print(f"\nDelta vs best baseline:  R@10: {delta_r10:+.2f}%   R@50: {delta_r50:+.2f}%")

## 7. Generate Submission

In [ ]:
TOP_K = 10
test_results = []

for item in tqdm(test_queries, desc="Generating test submissions"):
    query = item["query"]
    ranked = optimized_retrieve(query, top_k=TOP_K)

    retrievals = []
    for idx, score in ranked:
        retrievals.append({
            "id": doc_ids[idx],
            "text": doc_texts[idx],
            "title": doc_titles[idx],
        })

    test_results.append({
        "query_id": item["query_id"],
        "query": query,
        "retrievals": retrievals,
    })

print(f"Generated {len(test_results)} test results")

In [ ]:
OUTPUT_FILE = "test_queries_results.json"

with open(OUTPUT_FILE, "w") as f:
    json.dump(test_results, f, indent=2)
print(f"Saved {len(test_results)} results to {OUTPUT_FILE}")

# Preview
print(f"\nSample:")
print(json.dumps(test_results[0], indent=2, default=str)[:600])